# Article-Aware Retrieval Improvement for Turkish Legal RAG

This notebook improves the best previous pipeline by adding article-number-aware retrieval.

Previous best setup:
- Hybrid retrieval
- Turkish BGE reranker fusion
- Improved legal prompt
- Mistral-7B-Instruct

New setup:
- Hybrid retrieval
- Article number bonus
- Turkish BGE reranker fusion
- Improved legal prompt
- Mistral-7B-Instruct

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > T4 GPU seç.")

CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -q -U faiss-cpu sentence-transformers rank-bm25 transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


In [4]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [5]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

data_path = f"{project_path}/data"
processed_path = f"{project_path}/data/processed"
faiss_path = f"{project_path}/outputs/faiss"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Processed path:", processed_path)
print("FAISS path:", faiss_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
FAISS path: /content/drive/MyDrive/turkish_legal_rag/outputs/faiss
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [6]:
DATASET_CONFIG = {
    "dataset_name": "current_turkish_legal_dataset",

    "corpus_path": f"{processed_path}/retrieval_corpus.csv",
    "qa_path": f"{processed_path}/test_qa.csv",

    "text_column": "chunk_text",
    "question_column": "question",
    "answer_column": "answer",

    "source_column": "source",
    "chunk_id_column": "chunk_id",

    "faiss_index_path": f"{faiss_path}/baseline_faiss.index"
}

DATASET_CONFIG

{'dataset_name': 'current_turkish_legal_dataset',
 'corpus_path': '/content/drive/MyDrive/turkish_legal_rag/data/processed/retrieval_corpus.csv',
 'qa_path': '/content/drive/MyDrive/turkish_legal_rag/data/processed/test_qa.csv',
 'text_column': 'chunk_text',
 'question_column': 'question',
 'answer_column': 'answer',
 'source_column': 'source',
 'chunk_id_column': 'chunk_id',
 'faiss_index_path': '/content/drive/MyDrive/turkish_legal_rag/outputs/faiss/baseline_faiss.index'}

In [7]:
def load_table(path):
    path = str(path)

    if path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    elif path.endswith(".pkl") or path.endswith(".pickle"):
        with open(path, "rb") as f:
            return pickle.load(f)
    else:
        raise ValueError(f"Unsupported file format: {path}")

In [8]:
chunks_df = load_table(DATASET_CONFIG["corpus_path"])
test_qa_df = load_table(DATASET_CONFIG["qa_path"])

print("Chunks shape:", chunks_df.shape)
print("QA shape:", test_qa_df.shape)

display(chunks_df.head())
display(test_qa_df.head())

Chunks shape: (3775, 5)
QA shape: (1500, 2)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."
2,"Anayasa madde 172'ye göre, devletin sanayi ve ...","Anayasa madde 172'ye göre, devlet, sanayi ve t..."
3,"Anayasa madde 158, uyuşmazlık mahkemesi'nin ku...","Anayasa madde 158'e göre, uyuşmazlık mahkemesi..."
4,"Bir grup avukat, Türkiye Büyük Millet Meclisi ...","Anayasanın 94. Maddesi, Türkiye Büyük Millet M..."


In [9]:
TEXT_COL = DATASET_CONFIG["text_column"]
QUESTION_COL = DATASET_CONFIG["question_column"]
ANSWER_COL = DATASET_CONFIG["answer_column"]

SOURCE_COL = DATASET_CONFIG.get("source_column", "source")
CHUNK_ID_COL = DATASET_CONFIG.get("chunk_id_column", "chunk_id")

if CHUNK_ID_COL not in chunks_df.columns:
    chunks_df[CHUNK_ID_COL] = [f"chunk_{i:06d}" for i in range(len(chunks_df))]

if SOURCE_COL not in chunks_df.columns:
    chunks_df[SOURCE_COL] = DATASET_CONFIG["dataset_name"]

print("Text column:", TEXT_COL)
print("Question column:", QUESTION_COL)
print("Answer column:", ANSWER_COL)
print("Source column:", SOURCE_COL)
print("Chunk ID column:", CHUNK_ID_COL)

Text column: chunk_text
Question column: question
Answer column: answer
Source column: source
Chunk ID column: chunk_id


In [10]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [11]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

index = faiss.read_index(DATASET_CONFIG["faiss_index_path"])

print("Embedding model loaded:", embedding_model_name)
print("FAISS vectors:", index.ntotal)
print("Chunks:", len(chunks_df))

if index.ntotal != len(chunks_df):
    print("WARNING: FAISS index size does not match corpus size.")
else:
    print("FAISS index and corpus size match.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
FAISS vectors: 3775
Chunks: 3775
FAISS index and corpus size match.


In [12]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]

In [13]:
tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df[TEXT_COL].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

BM25 ready.


In [14]:
def extract_article_reference(query):
    """
    Extracts article number from Turkish legal questions.
    Handles examples like:
    - Anayasa madde 63
    - Anayasanın 101. Maddesi
    - Geçici madde 20
    - 108. Madde
    """
    q = str(query).lower()

    is_temporary = "geçici madde" in q or "gecici madde" in q

    patterns = [
        r"geçici\s+madde\s+(\d+)",
        r"gecici\s+madde\s+(\d+)",
        r"madde\s*(\d+)",
        r"(\d+)\.\s*madde",
        r"(\d+)\s*inci\s*madde",
        r"(\d+)\s*ıncı\s*madde",
        r"(\d+)\s*uncu\s*madde",
        r"(\d+)\s*üncü\s*madde",
        r"(\d+)\.\s*maddesi",
        r"(\d+)\s*maddesi"
    ]

    for pattern in patterns:
        match = re.search(pattern, q)
        if match:
            return {
                "article_number": match.group(1),
                "is_temporary": is_temporary
            }

    return None

In [28]:
def article_match_bonus(query, chunk_text):
    ref = extract_article_reference(query)

    if ref is None:
        return 0.0

    article_no = ref["article_number"]
    is_temporary = ref["is_temporary"]

    text = str(chunk_text).lower()

    # Köşeli parantez içindeki referansları madde eşleşmesi sayma.
    # Örn: [101] dipnot/referans olabilir, Madde 101 değildir.
    if re.search(rf"\[{article_no}\]", text):
        text_without_brackets = re.sub(r"\[[^\]]+\]", " ", text)
    else:
        text_without_brackets = text

    text = text_without_brackets

    if is_temporary:
        patterns = [
            rf"geçici\s+madde\s+{article_no}\b",
            rf"gecici\s+madde\s+{article_no}\b",
            rf"geçici\s+{article_no}\.\s*madde",
            rf"gecici\s+{article_no}\.\s*madde",
            rf"geçici\s+{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+madde",
            rf"gecici\s+{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+madde"
        ]
    else:
        patterns = [
            rf"madde\s+{article_no}\b",
            rf"madde\s*{article_no}\s*[–-]",
            rf"{article_no}\.\s*madde",
            rf"{article_no}\s*maddesi",
            rf"{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+madde",
            rf"{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+maddedeki",
            rf"{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+maddede",
            rf"{article_no}\s*(inci|ıncı|uncu|üncü|nci|ncı|ncu|ncü)\s+maddenin"
        ]

    for pattern in patterns:
        if re.search(pattern, text):
            return 1.0

    return 0.0

In [29]:
test_article_queries = [
    "Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?",
    "Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?",
    "Geçici madde 20 ne zaman eklendi?",
    "Cumhurbaşkanının görev süresi kaç yıldır?"
]

for q in test_article_queries:
    print(q, "->", extract_article_reference(q))

Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır? -> {'article_number': '63', 'is_temporary': False}
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir? -> {'article_number': '101', 'is_temporary': False}
Geçici madde 20 ne zaman eklendi? -> {'article_number': '20', 'is_temporary': True}
Cumhurbaşkanının görev süresi kaç yıldır? -> None


In [30]:
USE_SOURCE_FILTER = True
USE_ARTICLE_AWARE_BONUS = True
ARTICLE_BONUS_WEIGHT = 0.15

In [31]:
def detect_source_filter(query):
    q = str(query).lower()

    if (
        "anayasa" in q
        or "anayasanın" in q
        or "anayasa'nın" in q
        or "geçici madde" in q
        or "gecici madde" in q
    ):
        return "Türkiye Cumhuriyeti Anayasası"

    return None

In [32]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())

In [33]:
def hybrid_retrieve_top_k_article_aware(
    query,
    model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5,
    article_bonus_weight=0.25
):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    article_bonuses = np.array([
        article_match_bonus(query, chunks_df.iloc[i][TEXT_COL])
        for i in range(len(chunks_df))
    ], dtype=np.float32)

    if USE_ARTICLE_AWARE_BONUS:
        final_scores = hybrid_scores + article_bonus_weight * article_bonuses
    else:
        final_scores = hybrid_scores

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx][CHUNK_ID_COL],
            "source": chunks_df.iloc[idx][SOURCE_COL],
            "score": float(final_scores[idx]),
            "hybrid_score": float(hybrid_scores[idx]),
            "article_bonus": float(article_bonuses[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx][TEXT_COL]
        })

    return results

In [34]:
def hybrid_retrieve_top_k_filtered_article_aware(
    query,
    model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5,
    article_bonus_weight=0.25
):
    source_filter = detect_source_filter(query) if USE_SOURCE_FILTER else None

    candidate_df = chunks_df.copy()

    if source_filter is not None and SOURCE_COL in candidate_df.columns:
        filtered_df = candidate_df[
            candidate_df[SOURCE_COL].astype(str).str.lower() == source_filter.lower()
        ].reset_index(drop=True)

        if len(filtered_df) > 0:
            candidate_df = filtered_df

    candidate_texts = candidate_df[TEXT_COL].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    return hybrid_retrieve_top_k_article_aware(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha,
        article_bonus_weight=article_bonus_weight
    )

In [35]:
debug_questions = [
    "Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?",
    "Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?",
    "Anayasanın 13. Maddesi'ne aykırı mıdır?",
    "Geçici madde 20 ne zaman eklendi?",
    "Cumhurbaşkanının görev süresi kaç yıldır?"
]

for q in debug_questions:
    print("=" * 120)
    print("QUESTION:", q)
    print("ARTICLE REF:", extract_article_reference(q))

    results = hybrid_retrieve_top_k_filtered_article_aware(
        q,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    for r in results:
        print("-" * 80)
        print("Rank:", r["rank"])
        print("Chunk ID:", r["chunk_id"])
        print("Source:", r["source"])
        print("Final score:", r["score"])
        print("Hybrid score:", r["hybrid_score"])
        print("Article bonus:", r["article_bonus"])
        print(r["chunk_text"][:500])

QUESTION: Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?
ARTICLE REF: {'article_number': '63', 'is_temporary': False}
--------------------------------------------------------------------------------
Rank: 1
Chunk ID: chunk_000495
Source: Türkiye Cumhuriyeti Anayasası
Final score: 1.149999976158142
Hybrid score: 1.0
Article bonus: 1.0
Madde 63 – Devlet, tarih, kültür ve tabiat varlıklarının ve değerlerinin korunmasını sağlar, bu amaçla destekleyici ve teşvik edici tedbirleri alır.
--------------------------------------------------------------------------------
Rank: 2
Chunk ID: chunk_000494
Source: Türkiye Cumhuriyeti Anayasası
Final score: 0.6392641067504883
Hybrid score: 0.6392641067504883
Article bonus: 0.0
Madde 62 – Devlet, yabancı ülkelerde çalışan Türk vatandaşlarının aile birliğinin, çocuklarının eğitiminin, kültürel ihtiyaçlarının ve sosyal güvenliklerinin sağlanması, anavatanla bağlarının korunması ve yurda dönüşlerinde yardımcı olunma

In [36]:
search_terms = [
    "Geçici Madde 20",
    "geçici madde 20",
    "Geçici 20",
    "20 Mayıs 2016",
    "20/5/2016",
    "20.05.2016",
    "2016"
]

for term in search_terms:
    print("=" * 100)
    print("SEARCH TERM:", term)

    matches = chunks_df[
        chunks_df[TEXT_COL].astype(str).str.lower().str.contains(term.lower(), na=False)
    ]

    print("Match count:", len(matches))

    for _, row in matches.head(10).iterrows():
        print("-" * 80)
        print(row[CHUNK_ID_COL], row[SOURCE_COL])
        print(str(row[TEXT_COL])[:700])

SEARCH TERM: Geçici Madde 20
Match count: 0
SEARCH TERM: geçici madde 20
Match count: 0
SEARCH TERM: Geçici 20
Match count: 1
--------------------------------------------------------------------------------
chunk_003745 Türkiye Cumhuriyeti İş Kanunu
31/7/2008-5797/10 md.) Bu fıkrada düzenlenen teşvik, kamu idareleri hariç 506 sayılı Kanun kapsamındaki sigortalılara ilişkin matrah ve oranlar üzerinden olmak üzere, 506 sayılı Kanunun geçici 20 nci maddesi kapsamındaki sandıkların statülerine tabi personeli için de uygulanır. Bu fıkranın uygulanmasına ilişkin usul ve esaslar Maliye Bakanlığı ile Çalışma ve Sosyal Güvenlik Bakanlığı ve Hazine Müsteşarlığı tarafından müştereken belirlenir.[13]
(Değişik yedinci fıkra: 11/10/2011-KHK-665/28 md.) Bu maddeye aykırılık hallerinde 101 inci madde uyarınca tahsil edilecek cezalar, engellilerin ve eski hükümlülerin kendi işini kurmaları, engellinin iş bulmasını sağlayacak destek teknolojileri, engel
SEARCH TERM: 20 Mayıs 2016
Match count: 0
SEARCH T

## Corpus Coverage Observation

The query "Geçici madde 20 ne zaman eklendi?" was checked in the retrieval corpus.

Exact searches for:
- "Geçici Madde 20"
- "geçici madde 20"
- "20 Mayıs 2016"
- "20/5/2016"
- "20.05.2016"

did not return the expected answer source.

Therefore, this sample is treated as a corpus coverage problem. The RAG system cannot reliably answer this question if the required answer is not present in the retrieval corpus.

In [37]:
turkish_reranker_model_name = "seroe/bge-reranker-v2-m3-turkish-triplet"

turkish_reranker = CrossEncoder(
    turkish_reranker_model_name,
    device="cpu",
    max_length=512
)

print("Turkish BGE reranker loaded:", turkish_reranker_model_name)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Turkish BGE reranker loaded: seroe/bge-reranker-v2-m3-turkish-triplet


In [38]:
def rerank_retrieved_chunks_turkish_bge(question, retrieved_results, top_k=None):
    pairs = [
        [question, item["chunk_text"]]
        for item in retrieved_results
    ]

    scores = turkish_reranker.predict(
        pairs,
        batch_size=4,
        show_progress_bar=False
    )

    scored_results = []

    for item, score in zip(retrieved_results, scores):
        scored_results.append({
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "original_rank": item["rank"],
            "original_hybrid_score": item.get("hybrid_score", item["score"]),
            "article_bonus": item.get("article_bonus", 0.0),
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "rerank_score": float(score),
            "chunk_text": item["chunk_text"]
        })

    scored_results = sorted(
        scored_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    if top_k is None:
        top_k = len(scored_results)

    final_results = []

    for new_rank, item in enumerate(scored_results[:top_k], start=1):
        item["rerank_rank"] = new_rank
        final_results.append(item)

    return final_results

In [39]:
def hybrid_retrieve_with_article_aware_turkish_bge_fusion(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3,
    article_bonus_weight=0.25
):
    candidates = hybrid_retrieve_top_k_filtered_article_aware(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha,
        article_bonus_weight=article_bonus_weight
    )

    reranked_all = rerank_retrieved_chunks_turkish_bge(
        question=query,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    rerank_rank_map = {
        item["chunk_id"]: item["rerank_rank"]
        for item in reranked_all
    }

    rerank_score_map = {
        item["chunk_id"]: item["rerank_score"]
        for item in reranked_all
    }

    fused_results = []

    for item in candidates:
        chunk_id = item["chunk_id"]

        original_rank = item["rank"]
        rerank_rank = rerank_rank_map.get(chunk_id, candidate_k + 1)

        hybrid_rank_score = 1 / original_rank
        rerank_rank_score = 1 / rerank_rank

        fusion_score = (
            hybrid_weight * hybrid_rank_score
            + rerank_weight * rerank_rank_score
        )

        fused_results.append({
            "rank": None,
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "fusion_score": float(fusion_score),
            "original_rank": original_rank,
            "rerank_rank": rerank_rank,
            "rerank_score": float(rerank_score_map.get(chunk_id, 0.0)),
            "article_bonus": item.get("article_bonus", 0.0),
            "original_hybrid_score": item.get("hybrid_score", item["score"]),
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "chunk_text": item["chunk_text"]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    final_results = []

    for new_rank, item in enumerate(fused_results[:final_k], start=1):
        item["rank"] = new_rank
        final_results.append(item)

    return final_results

In [40]:
CANDIDATE_K = 10
FINAL_CONTEXT_K = 3
ALPHA = 0.5

HYBRID_WEIGHT = 0.7
RERANK_WEIGHT = 0.3

ARTICLE_BONUS_WEIGHT = 0.25

In [41]:
sample_question = "Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?"

article_fusion_results = hybrid_retrieve_with_article_aware_turkish_bge_fusion(
    sample_question,
    embedding_model,
    index,
    chunks_df,
    bm25,
    candidate_k=CANDIDATE_K,
    final_k=FINAL_CONTEXT_K,
    alpha=ALPHA,
    hybrid_weight=HYBRID_WEIGHT,
    rerank_weight=RERANK_WEIGHT,
    article_bonus_weight=ARTICLE_BONUS_WEIGHT
)

for r in article_fusion_results:
    print("=" * 80)
    print("Rank:", r["rank"])
    print("Chunk ID:", r["chunk_id"])
    print("Original rank:", r["original_rank"])
    print("Rerank rank:", r["rerank_rank"])
    print("Article bonus:", r["article_bonus"])
    print("Fusion score:", r["fusion_score"])
    print("Source:", r["source"])
    print(r["chunk_text"][:500])

Rank: 1
Chunk ID: chunk_000495
Original rank: 1
Rerank rank: 1
Article bonus: 1.0
Fusion score: 1.0
Source: Türkiye Cumhuriyeti Anayasası
Madde 63 – Devlet, tarih, kültür ve tabiat varlıklarının ve değerlerinin korunmasını sağlar, bu amaçla destekleyici ve teşvik edici tedbirleri alır.
Rank: 2
Chunk ID: chunk_000494
Original rank: 2
Rerank rank: 2
Article bonus: 0.0
Fusion score: 0.5
Source: Türkiye Cumhuriyeti Anayasası
Madde 62 – Devlet, yabancı ülkelerde çalışan Türk vatandaşlarının aile birliğinin, çocuklarının eğitiminin, kültürel ihtiyaçlarının ve sosyal güvenliklerinin sağlanması, anavatanla bağlarının korunması ve yurda dönüşlerinde yardımcı olunması için gereken tedbirleri alır. X I. Tarih, kültür ve tabiat varlıklarının korunması
Rank: 3
Chunk ID: chunk_000120
Original rank: 3
Rerank rank: 9
Article bonus: 0.0
Fusion score: 0.26666666666666666
Source: Türkiye Cumhuriyeti Anayasası
Madde 129 – Memurlar ve diğer kamu görevlileri Anayasa ve kanunlara sadık kalarak faaliyette bul

In [42]:
def build_improved_legal_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Bağlam {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
Sen Türk hukuk metinleri üzerinde çalışan dikkatli bir RAG soru-cevap asistanısın.

Görevin:
Sadece verilen bağlamları kullanarak soruya kısa, net ve hukuki olarak doğru cevap vermek.

Zorunlu kurallar:
1. Cevabı yalnızca verilen bağlamlara dayandır.
2. Bağlamda açıkça bulunmayan bilgiyi uydurma.
3. Soru bir süre, tarih, sayı veya madde soruyorsa sadece ilgili değeri ve kısa açıklamasını ver.
4. Soru "aykırı mıdır", "çelişir mi", "uygun mudur" gibi bir değerlendirme soruyorsa cevaba mutlaka "Evet" veya "Hayır" ile başla.
5. Cevapta "Bağlam", "Context", "verilen metne göre" gibi ifadeler kullanma.
6. Cevap en fazla iki kısa cümle olmalı.
7. Alternatif cevap, yeni soru, örnek soru, başlık veya açıklama bölümü üretme.
8. Cevabı verdikten sonra dur.
9. Eğer bağlamda cevap yoksa sadece şunu yaz: "Verilen bağlamda bu sorunun cevabı bulunamamaktadır."

Bağlamlar:
{context_text}

Soru:
{question}

Kısa ve doğrudan cevap:
"""
    return prompt.strip()

In [43]:
from huggingface_hub import login
login()

In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("LLM loaded:", llm_model_name)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded: mistralai/Mistral-7B-Instruct-v0.2


In [45]:
def generate_answer(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    markers = [
        "Kısa ve doğrudan cevap:",
        "Kısa cevap:",
        "Answer:"
    ]

    for marker in markers:
        if marker in generated_text:
            generated_text = generated_text.split(marker)[-1].strip()

    return generated_text.strip()

In [46]:
def clean_generated_answer(text):
    text = str(text).strip()

    start_markers = [
        "Kısa ve doğrudan cevap:",
        "Kısa cevap:",
        "Answer:"
    ]

    for marker in start_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    stop_markers = [
        "\nBaşka bir şekilde:",
        "\nCevap:",
        "\nSoru:",
        "\nQuestion:",
        "\nAlternatif",
        "\nAçıklama:",
        "\nDetaylı cevap:",
        "\nÖrnek:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    unwanted = [
        "Context 1", "Context 2", "Context 3",
        "Bağlam 1", "Bağlam 2", "Bağlam 3"
    ]

    for u in unwanted:
        text = text.replace(u, "")

    text = re.sub(r"\(\s*\)", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [47]:
def postprocess_answer_by_question_type(question, answer):
    q = str(question).lower()
    answer = str(answer).strip()

    short_answer_triggers = [
        "kaç",
        "ne zaman",
        "kime aittir",
        "en fazla kaç"
    ]

    if any(trigger in q for trigger in short_answer_triggers):
        sentences = re.split(r"(?<=[.!?])\s+", answer)
        if len(sentences) > 0:
            return sentences[0].strip()

    return answer

In [48]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)

In [49]:
article_aware_small_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_with_article_aware_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    article_aware_small_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_article_bonus": retrieved[0]["article_bonus"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_fusion_score": retrieved[0]["fusion_score"]
    })

article_aware_small_results_df = pd.DataFrame(article_aware_small_results)
article_aware_small_results_df

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_fusion_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,0.0,1,3,0.8
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0


In [50]:
for i, row in article_aware_small_results_df.iterrows():
    print("=" * 100)
    print("INDEX:", i)
    print("QUESTION:", row["question"])
    print("EXPECTED:", row["expected_answer"])
    print("GENERATED:", row["clean_generated_answer"])
    print("TOP1:", row["top1_chunk_id"], row["top1_source"], "bonus:", row["top1_article_bonus"])

INDEX: 0
QUESTION: Egemenlik kime aittir?
EXPECTED: Egemenlik kayıtsız şartsız Milletindir.
GENERATED: Egemenlik Türk Milleti'ne aittir.
TOP1: chunk_000269 Türkiye Cumhuriyeti Anayasası bonus: 0.0
INDEX: 1
QUESTION: Türkiye Cumhuriyetinin yönetim şekli nedir?
EXPECTED: Türkiye Devleti bir Cumhuriyettir.
GENERATED: Türkiye Cumhuriyetinin yönetim şekli bölünmez bir bütünlük olmasına dayalı bir halk ve toprak demokratik republiktir.
TOP1: chunk_000000 Türkiye Cumhuriyeti Anayasası bonus: 0.0
INDEX: 2
QUESTION: Cumhurbaşkanının görev süresi kaç yıldır?
EXPECTED: Cumhurbaşkanının görev süresi beş yıldır.
GENERATED: Cumhurbaşkanının görev süresi beş yıldır.
TOP1: chunk_000043 Türkiye Cumhuriyeti Anayasası bonus: 0.0
INDEX: 3
QUESTION: Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?
EXPECTED: Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
GENERATED: Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
TOP1: chunk_000043 Türkiye Cumhuriyeti Anayasası bonus: 0.0


In [51]:
manual_scores_article_aware_small = [
    1.0,  # Egemenlik Türk Milleti'ne aittir: anlam doğru.
    0.0,  # Yönetim şekli cevabı yanlış/bozuk.
    1.0,  # Görev süresi doğru.
    1.0   # En fazla iki defa doğru.
]

article_aware_small_results_df["manual_score"] = manual_scores_article_aware_small

article_aware_small_score = article_aware_small_results_df["manual_score"].mean()

print("Article-Aware Retrieval 4-question score:", article_aware_small_score)
article_aware_small_results_df

Article-Aware Retrieval 4-question score: 0.75


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_fusion_score,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,0.0,1,3,0.8,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0,1.0


In [52]:
article_aware_small_results_df.to_csv(
    f"{metrics_path}/article_aware_turkish_bge_4question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Article-Aware Retrieval + Turkish BGE Reranker Fusion - 4 Question Sanity Evaluation",
    "manual_accuracy": article_aware_small_score,
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT,
    "article_bonus_weight": ARTICLE_BONUS_WEIGHT
}]).to_csv(
    f"{metrics_path}/article_aware_turkish_bge_4question_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Article-aware 4-question results saved.")

Article-aware 4-question results saved.


In [53]:
article_aware_test_results = []

for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df)):
    question = row[QUESTION_COL]
    expected_answer = row[ANSWER_COL]

    retrieved = hybrid_retrieve_with_article_aware_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    article_aware_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_article_bonus": retrieved[0]["article_bonus"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_rerank_score": retrieved[0]["rerank_score"],
        "top1_fusion_score": retrieved[0]["fusion_score"],
        "retrieved_contexts": "\n\n".join(contexts)
    })

article_aware_test_results_df = pd.DataFrame(article_aware_test_results)
article_aware_test_results_df.head()

100%|██████████| 20/20 [06:23<00:00, 19.19s/it]


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000537,Türkiye Cumhuriyeti Anayasası,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,1.0,1,1,0.023523,1.0000,Cumhurbaşkanlığı seçiminde birinci oylamada ge...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000274,Türkiye Cumhuriyeti Anayasası,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",1.0,1,1,0.027961,1.0000,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1.0,1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 2001-10-03 tarihinde eklendi.,Geçici madde 20 2001-10-03 tarihinde eklendi.,chunk_000475,Türkiye Cumhuriyeti Anayasası,Madde 52 – (Mülga: 23/7/1995-4121/3 md.) V I. ...,0.0,1,8,0.001482,0.7375,Madde 52 – (Mülga: 23/7/1995-4121/3 md.) V I. ...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde içerikleri ulaşım araçları...","Evet, TCK 121 madde içerikleri ulaşım araçları...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,0.0,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [54]:
article_aware_test_results_df.to_csv(
    f"{metrics_path}/article_aware_turkish_bge_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Article-aware test generation results saved.")

Article-aware test generation results saved.


In [55]:
for i, row in article_aware_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 SOURCE:", row["top1_source"])
    print("TOP1 ARTICLE BONUS:", row["top1_article_bonus"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, önceki seçimde gerekli çoğunluğun sağlanamaması halinde ikinci seçim yapılması ve seçimlerin geriye bırakılması

TOP1 CHUNK ID: chunk_000537
TOP1 SOURCE: Türkiye Cumhuriyeti Anayasası
TOP1 ARTICLE BONUS: 1.0
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 1
TOP1 FUSION SCORE: 1.0
INDEX: 1

QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, herkesin dil, ırk, renk, cinsiyet, siyasi düşünce, felsefi inanç, din, mezhep ve benzeri sebeplerle ayırım gözetilmek

In [56]:
article_aware_test_results_df["is_valid_sample"] = True

# Önceki deneylerle aynı evaluation setup:
# index 19 invalid sample olarak skora dahil edilmiyor.
article_aware_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_article_aware = [
    0.0,  # 0 - Article 101: beklenen seçilme şartları/uygulama tartışması değil, seçim prosedürü anlatıyor.
    0.5,  # 1 - Eşitlik ilkesini yakalıyor ama cevap kesilmiş ve aykırılık net değil.
    0.5,  # 2 - Yaşama hakkı bağlamı var ama "Hayır" diyerek sonucu ters veriyor.
    0.0,  # 3 - Geçici madde 20 tarihi yanlış; corpus coverage problemi de var.
    0.5,  # 4 - "Sabit değildir" fikri var ama bağlam/gerekçe yanlış ve bilirkişi raporu yok.
    0.0,  # 5 - Cumhurbaşkanı yemini yerine alakasız yemin/yazman cevabı.
    0.0,  # 6 - "Anayasa 122 yoktur" diyor, yanlış.
    1.0,  # 7 - Yedi gün doğru.
    0.5,  # 8 - "Evet" doğru yön ama açıklama karışık/eksik.
    0.0,  # 9 - 108. madde ve Silahlı Kuvvetler istisnasını doğru vermiyor.
    1.0,  # 10 - 13. madde için aykırıdır ve kanunla sınırlama gerekçesi doğru.
    0.0,  # 11 - KVKK'daki ilgili kişi tanımı yanlış.
    0.5,  # 12 - AYM kararlarının kesin uygulanması fikri var ama kesinlik anlamı eksik.
    1.0,  # 13 - Madde 63 doğru.
    0.5,  # 14 - Madde 47 kısmen doğru ama eksik/karışık.
    0.5,  # 15 - "Kanunla düzenlenir" kısmen doğru ama KVKK ve diğer kanunlar net değil.
    0.5,  # 16 - Kanunla düzenlenir ana fikri doğru ama detaylar eksik.
    0.0,  # 17 - 115 yerine 109 demiş, yanlış.
    0.5,  # 18 - Bilgi Edinme Değerlendirme Kurulu fikri var ama cevap eksik/kesilmiş.
    0.0   # 19 - Invalid sample, skora katılmayacak.
]

article_aware_test_results_df["manual_score"] = manual_scores_article_aware

valid_article_aware_df = article_aware_test_results_df[
    article_aware_test_results_df["is_valid_sample"] == True
]

article_aware_test_score = valid_article_aware_df["manual_score"].mean()

print("Article-Aware Retrieval + Turkish BGE Reranker Fusion RAG Test Score:", article_aware_test_score)
print("Valid sample count:", len(valid_article_aware_df))
print("Total sample count:", len(article_aware_test_results_df))

Article-Aware Retrieval + Turkish BGE Reranker Fusion RAG Test Score: 0.39473684210526316
Valid sample count: 19
Total sample count: 20


In [57]:
article_aware_test_results_df.to_csv(
    f"{metrics_path}/article_aware_turkish_bge_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Article-Aware Retrieval + Turkish BGE Reranker Fusion + Improved Prompt",
    "manual_accuracy": article_aware_test_score,
    "valid_sample_count": len(valid_article_aware_df),
    "total_sample_count": len(article_aware_test_results_df),
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT,
    "article_bonus_weight": ARTICLE_BONUS_WEIGHT
}]).to_csv(
    f"{metrics_path}/article_aware_turkish_bge_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Article-aware scored results saved.")

Article-aware scored results saved.


In [58]:
test_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Base prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "Strict Prompt RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "FlashRank Fusion Reranker RAG",
        "retrieval_setup": "Hybrid top-10 + FlashRank reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.184211
    },
    {
        "method": "Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.315789
    },
    {
        "method": "Improved Prompt + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": 0.394737
    },
    {
        "method": "Article-Aware Retrieval + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + article bonus + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": article_aware_test_score
    }
])

test_comparison_df

,method,retrieval_setup,prompt,manual_accuracy
0,Base RAG,"Hybrid retrieval top-5, context top-3",Base prompt,0.263158
1,Strict Prompt RAG,"Hybrid retrieval top-5, context top-3",Strict prompt,0.263158
2,FlashRank Fusion Reranker RAG,Hybrid top-10 + FlashRank reranker + rank fusi...,Strict prompt,0.184211
3,Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Strict prompt,0.315789
4,Improved Prompt + Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Improved legal prompt,0.394737
5,Article-Aware Retrieval + Turkish BGE Reranker...,Hybrid top-10 + article bonus + Turkish BGE re...,Improved legal prompt,0.394737


In [59]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_article_aware_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Article-aware comparison saved.")

Article-aware comparison saved.


## Source-Aware Routing Experiment

The article-aware retrieval experiment showed that article number matching can promote some relevant chunks, but the final manual score did not improve.

In this experiment, we add source-aware routing before retrieval. The goal is to route questions to the most likely legal source, such as:
- Turkish Constitution
- Turkish Penal Code
- Personal Data Protection Law
- Information Access Law
- Criminal Procedure Code

This may reduce wrong-context retrieval errors.

In [60]:
source_counts = chunks_df[SOURCE_COL].value_counts()
source_counts

,count
source,
Türk Medeni Kanunu,1109
Ceza Muhakemesi Kanunu,765
Türk Borçlar Kanunu,639
Türkiye Cumhuriyeti Anayasası,561
Türk Ceza Kanunu,389
Türkiye Cumhuriyeti İş Kanunu,186
Türk Bayrağı Tüzüğü,66
Bilgi Edinme Kanunu,60


In [61]:
def normalize_for_match(text):
    text = str(text).lower()
    text = text.replace("ı", "i")
    text = text.replace("ğ", "g")
    text = text.replace("ü", "u")
    text = text.replace("ş", "s")
    text = text.replace("ö", "o")
    text = text.replace("ç", "c")
    text = re.sub(r"\s+", " ", text).strip()
    return text


available_sources = chunks_df[SOURCE_COL].dropna().astype(str).unique().tolist()

print("Available sources:")
for s in available_sources:
    print("-", s)


def resolve_source_name(target_name):
    target_norm = normalize_for_match(target_name)

    for source in available_sources:
        if normalize_for_match(source) == target_norm:
            return source

    for source in available_sources:
        source_norm = normalize_for_match(source)
        if target_norm in source_norm or source_norm in target_norm:
            return source

    return None

Available sources:
- Türkiye Cumhuriyeti Anayasası
- Bilgi Edinme Kanunu
- Ceza Muhakemesi Kanunu
- Türk Medeni Kanunu
- Türk Bayrağı Tüzüğü
- Türk Borçlar Kanunu
- Türk Ceza Kanunu
- Türkiye Cumhuriyeti İş Kanunu


In [62]:
USE_SOURCE_FILTER = True
USE_SOURCE_AWARE_ROUTING = True
MIN_FILTERED_ROWS = 5


def detect_source_filter(query):
    q = normalize_for_match(query)

    # KVKK-related questions.
    # Important: KVKK source is not available in the current corpus.
    if (
        "kisisel veri" in q
        or "kisisel veriler" in q
        or "kvkk" in q
        or "ilgili kisi" in q
        or "verisi islenen" in q
    ):
        kvkk_source = resolve_source_name("Kişisel Verilerin Korunması Kanunu")
        return kvkk_source  # current corpus için büyük ihtimalle None dönecek

    if (
        "bilgi edinme" in q
        or "bilgi edinme hakki" in q
        or "bilgi edinme degerlendirme kurulu" in q
    ):
        return resolve_source_name("Bilgi Edinme Kanunu")

    if (
        "tck" in q
        or "turk ceza kanunu" in q
        or "turk ceza" in q
    ):
        return resolve_source_name("Türk Ceza Kanunu")

    if (
        "cmk" in q
        or "ceza muhakemesi" in q
    ):
        return resolve_source_name("Ceza Muhakemesi Kanunu")

    if (
        "is kanunu" in q
        or "isci" in q
        or "isveren" in q
    ):
        return resolve_source_name("Türkiye Cumhuriyeti İş Kanunu")

    if (
        "medeni kanun" in q
        or "turk medeni" in q
    ):
        return resolve_source_name("Türk Medeni Kanunu")

    if (
        "borclar kanunu" in q
        or "turk borclar" in q
    ):
        return resolve_source_name("Türk Borçlar Kanunu")

    # Constitutional/legal institution questions.
    if (
        "anayasa" in q
        or "anayasanin" in q
        or "cumhurbaskani" in q
        or "yasama dokunulmazligi" in q
        or "anayasa mahkemesi" in q
        or "devlet denetleme kurulu" in q
        or "ekonomik ve sosyal konsey" in q
        or "temel hak" in q
        or "silahli kuvvetler" in q
    ):
        return resolve_source_name("Türkiye Cumhuriyeti Anayasası")

    return None

In [63]:
routing_debug_questions = [
    "Videoda TCK 121 ihlali sabit değil mi?",
    "İlgili kişi ne demektir?",
    "Kişisel veriler hangi mevzuata göre işlenir?",
    "Bilgi edinme hakkının ihlali durumunda ne yapılabilir",
    "Anayasanın 108. Maddesi ile nasıl çelişir?",
    "Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?",
    "Bir siyasi parti, çalışma şartlarının belirli durumlarda devlet tarafından keyfi olarak düzenlenebileceğini savunmaktadır. Bu durum Anayasanın 50. Maddesi'ne aykırı mıdır?"
]

for q in routing_debug_questions:
    print("=" * 100)
    print("QUESTION:", q)
    print("ROUTED SOURCE:", detect_source_filter(q))

QUESTION: Videoda TCK 121 ihlali sabit değil mi?
ROUTED SOURCE: Türk Ceza Kanunu
QUESTION: İlgili kişi ne demektir?
ROUTED SOURCE: None
QUESTION: Kişisel veriler hangi mevzuata göre işlenir?
ROUTED SOURCE: None
QUESTION: Bilgi edinme hakkının ihlali durumunda ne yapılabilir
ROUTED SOURCE: Bilgi Edinme Kanunu
QUESTION: Anayasanın 108. Maddesi ile nasıl çelişir?
ROUTED SOURCE: Türkiye Cumhuriyeti Anayasası
QUESTION: Anayasa madde 63'e göre, tarih, kültür ve tabiat varlıklarının korunması nasıl sağlanır?
ROUTED SOURCE: Türkiye Cumhuriyeti Anayasası
QUESTION: Bir siyasi parti, çalışma şartlarının belirli durumlarda devlet tarafından keyfi olarak düzenlenebileceğini savunmaktadır. Bu durum Anayasanın 50. Maddesi'ne aykırı mıdır?
ROUTED SOURCE: Türkiye Cumhuriyeti Anayasası


In [64]:
def hybrid_retrieve_top_k_filtered_article_aware(
    query,
    model,
    index,
    chunks_df,
    bm25,
    k=5,
    alpha=0.5,
    article_bonus_weight=0.15
):
    source_filter = detect_source_filter(query) if USE_SOURCE_FILTER else None

    candidate_df = chunks_df.copy()
    used_source_filter = None

    if source_filter is not None and SOURCE_COL in candidate_df.columns:
        filtered_df = candidate_df[
            candidate_df[SOURCE_COL].astype(str).str.lower() == str(source_filter).lower()
        ].reset_index(drop=True)

        if len(filtered_df) >= MIN_FILTERED_ROWS:
            candidate_df = filtered_df
            used_source_filter = source_filter
        else:
            candidate_df = chunks_df.copy()
            used_source_filter = None

    candidate_texts = candidate_df[TEXT_COL].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    results = hybrid_retrieve_top_k_article_aware(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha,
        article_bonus_weight=article_bonus_weight
    )

    for item in results:
        item["source_filter"] = used_source_filter

    return results

In [65]:
def hybrid_retrieve_with_article_aware_turkish_bge_fusion(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3,
    article_bonus_weight=0.15
):
    candidates = hybrid_retrieve_top_k_filtered_article_aware(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha,
        article_bonus_weight=article_bonus_weight
    )

    reranked_all = rerank_retrieved_chunks_turkish_bge(
        question=query,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    rerank_rank_map = {
        item["chunk_id"]: item["rerank_rank"]
        for item in reranked_all
    }

    rerank_score_map = {
        item["chunk_id"]: item["rerank_score"]
        for item in reranked_all
    }

    fused_results = []

    for item in candidates:
        chunk_id = item["chunk_id"]

        original_rank = item["rank"]
        rerank_rank = rerank_rank_map.get(chunk_id, candidate_k + 1)

        hybrid_rank_score = 1 / original_rank
        rerank_rank_score = 1 / rerank_rank

        fusion_score = (
            hybrid_weight * hybrid_rank_score
            + rerank_weight * rerank_rank_score
        )

        fused_results.append({
            "rank": None,
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "source_filter": item.get("source_filter", None),
            "fusion_score": float(fusion_score),
            "original_rank": original_rank,
            "rerank_rank": rerank_rank,
            "rerank_score": float(rerank_score_map.get(chunk_id, 0.0)),
            "article_bonus": item.get("article_bonus", 0.0),
            "original_hybrid_score": item.get("hybrid_score", item["score"]),
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "chunk_text": item["chunk_text"]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    final_results = []

    for new_rank, item in enumerate(fused_results[:final_k], start=1):
        item["rank"] = new_rank
        final_results.append(item)

    return final_results

In [66]:
source_debug_questions = [
    "Videoda TCK 121 ihlali sabit değil mi?",
    "İlgili kişi ne demektir?",
    "Kişisel veriler hangi mevzuata göre işlenir?",
    "Bilgi edinme hakkının ihlali durumunda ne yapılabilir",
    "Bir siyasi parti, çalışma şartlarının belirli durumlarda devlet tarafından keyfi olarak düzenlenebileceğini savunmaktadır. Bu durum Anayasanın 50. Maddesi'ne aykırı mıdır?"
]

for q in source_debug_questions:
    print("=" * 120)
    print("QUESTION:", q)
    print("ROUTED SOURCE:", detect_source_filter(q))

    results = hybrid_retrieve_top_k_filtered_article_aware(
        q,
        embedding_model,
        index,
        chunks_df,
        bm25,
        k=5,
        alpha=0.5,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    for r in results:
        print("-" * 80)
        print("Rank:", r["rank"])
        print("Chunk ID:", r["chunk_id"])
        print("Source:", r["source"])
        print("Used source filter:", r.get("source_filter", None))
        print("Final score:", r["score"])
        print("Hybrid score:", r["hybrid_score"])
        print("Article bonus:", r["article_bonus"])
        print(r["chunk_text"][:500])

QUESTION: Videoda TCK 121 ihlali sabit değil mi?
ROUTED SOURCE: Türk Ceza Kanunu
--------------------------------------------------------------------------------
Rank: 1
Chunk ID: chunk_003431
Source: Türk Ceza Kanunu
Used source filter: Türk Ceza Kanunu
Final score: 0.8763986825942993
Hybrid score: 0.8763986825942993
Article bonus: 0.0
ÜÇÜNCÜ KISIM
Topluma Karşı Suçlar ALTINCI BÖLÜM
Ulaşım Araçlarına veya Sabit Platformlara Karşı Suçlar
--------------------------------------------------------------------------------
Rank: 2
Chunk ID: chunk_003326
Source: Türk Ceza Kanunu
Used source filter: Türk Ceza Kanunu
Final score: 0.8650658130645752
Hybrid score: 0.8650658130645752
Article bonus: 0.0
Dilekçe hakkının kullanılmasının engellenmesi
MADDE 121. - (1) Kişinin belli bir hakkı kullanmak için yetkili kamu makamlarına verdiği dilekçenin hukukî bir neden olmaksızın kabul edilmemesi hâlinde, fail hakkında altı aya kadar hapis cezasına hükmolunur. Ayırımcılık
MADDE 122. - (1) Kişiler arasınd

In [67]:
source_aware_small_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_with_article_aware_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    source_aware_small_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_source_filter": retrieved[0].get("source_filter", None),
        "top1_article_bonus": retrieved[0]["article_bonus"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_fusion_score": retrieved[0]["fusion_score"]
    })

source_aware_small_results_df = pd.DataFrame(source_aware_small_results)
source_aware_small_results_df

,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_source_filter,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_fusion_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,None,0.0,1,1,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,None,0.0,1,3,0.8
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır. (Bağ...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0


In [68]:
for i, row in source_aware_small_results_df.iterrows():
    print("=" * 100)
    print("INDEX:", i)
    print("QUESTION:", row["question"])
    print("EXPECTED:", row["expected_answer"])
    print("GENERATED:", row["clean_generated_answer"])
    print("TOP1:", row["top1_chunk_id"], row["top1_source"])
    print("SOURCE FILTER:", row["top1_source_filter"])
    print("ARTICLE BONUS:", row["top1_article_bonus"])

INDEX: 0
QUESTION: Egemenlik kime aittir?
EXPECTED: Egemenlik kayıtsız şartsız Milletindir.
GENERATED: Egemenlik Türk Milleti'ne aittir.
TOP1: chunk_000269 Türkiye Cumhuriyeti Anayasası
SOURCE FILTER: None
ARTICLE BONUS: 0.0
INDEX: 1
QUESTION: Türkiye Cumhuriyetinin yönetim şekli nedir?
EXPECTED: Türkiye Devleti bir Cumhuriyettir.
GENERATED: Türkiye Cumhuriyetinin yönetim şekli bölünmez bir bütünlük olmasına dayalı bir halk ve toprak demokratik republiktir.
TOP1: chunk_000000 Türkiye Cumhuriyeti Anayasası
SOURCE FILTER: None
ARTICLE BONUS: 0.0
INDEX: 2
QUESTION: Cumhurbaşkanının görev süresi kaç yıldır?
EXPECTED: Cumhurbaşkanının görev süresi beş yıldır.
GENERATED: Cumhurbaşkanının görev süresi beş yıldır.
TOP1: chunk_000043 Türkiye Cumhuriyeti Anayasası
SOURCE FILTER: Türkiye Cumhuriyeti Anayasası
ARTICLE BONUS: 0.0
INDEX: 3
QUESTION: Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?
EXPECTED: Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
GENERATED: Bir kimse en fazla ik

In [69]:
manual_scores_source_aware_small = [
    1.0,
    0.0,
    1.0,
    1.0
]

source_aware_small_results_df["manual_score"] = manual_scores_source_aware_small

source_aware_small_score = source_aware_small_results_df["manual_score"].mean()

print("Source-Aware + Article-Aware 4-question score:", source_aware_small_score)
source_aware_small_results_df

Source-Aware + Article-Aware 4-question score: 0.75


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_source_filter,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_fusion_score,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,None,0.0,1,1,1.0,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,None,0.0,1,3,0.8,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır. (Bağ...,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,0.0,1,1,1.0,1.0


In [70]:
source_aware_small_results_df.to_csv(
    f"{metrics_path}/source_aware_article_aware_turkish_bge_4question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Source-Aware + Article-Aware Retrieval + Turkish BGE Reranker Fusion - 4 Question Sanity Evaluation",
    "manual_accuracy": source_aware_small_score,
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT,
    "article_bonus_weight": ARTICLE_BONUS_WEIGHT
}]).to_csv(
    f"{metrics_path}/source_aware_article_aware_turkish_bge_4question_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Source-aware 4-question results saved.")

Source-aware 4-question results saved.


In [71]:
source_aware_test_results = []

for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df)):
    question = row[QUESTION_COL]
    expected_answer = row[ANSWER_COL]

    retrieved = hybrid_retrieve_with_article_aware_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT,
        article_bonus_weight=ARTICLE_BONUS_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    source_aware_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_source_filter": retrieved[0].get("source_filter", None),
        "top1_context": retrieved[0]["chunk_text"],
        "top1_article_bonus": retrieved[0]["article_bonus"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_rerank_score": retrieved[0]["rerank_score"],
        "top1_fusion_score": retrieved[0]["fusion_score"],
        "retrieved_contexts": "\n\n".join(contexts)
    })

source_aware_test_results_df = pd.DataFrame(source_aware_test_results)
source_aware_test_results_df.head()

100%|██████████| 20/20 [05:51<00:00, 17.59s/it]


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_source_filter,top1_context,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000537,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,1.0,1,1,0.023523,1.00,Cumhurbaşkanlığı seçiminde birinci oylamada ge...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000274,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",1.0,1,1,0.027961,1.00,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1.0,1,1,0.385170,1.00,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,Geçici madde 20 13/5/1981 gün eklendi.,chunk_003745,Türkiye Cumhuriyeti İş Kanunu,None,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,1.0,1,1,0.438255,1.00,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...","Evet, TCK 121 madde hakkının kullanılmasının e...",chunk_003431,Türk Ceza Kanunu,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...,0.0,1,5,0.002067,0.76,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...


In [72]:
source_aware_test_results_df.to_csv(
    f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Source-aware article-aware generation results saved.")

Source-aware article-aware generation results saved.


In [73]:
for i, row in source_aware_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 SOURCE:", row["top1_source"])
    print("TOP1 SOURCE FILTER:", row["top1_source_filter"])
    print("TOP1 ARTICLE BONUS:", row["top1_article_bonus"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, önceki seçimde gerekli çoğunluğun sağlanamaması halinde ikinci seçim yapılması ve seçimlerin geriye bırakılması

TOP1 CHUNK ID: chunk_000537
TOP1 SOURCE: Türkiye Cumhuriyeti Anayasası
TOP1 SOURCE FILTER: Türkiye Cumhuriyeti Anayasası
TOP1 ARTICLE BONUS: 1.0
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 1
TOP1 FUSION SCORE: 1.0
INDEX: 1

QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, herkesin dil, ırk, renk, cinsiyet, siyasi düşünce, felsefi inanç, d

In [74]:
source_aware_test_results_df["is_valid_sample"] = True

# Önceki deneylerle aynı evaluation setup:
# index 19 invalid sample olarak skora dahil edilmiyor.
source_aware_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_source_aware = [
    0.0,  # 0 - Article 101: seçilme şartları/tartışma yerine seçim prosedürü.
    0.5,  # 1 - Eşitlik ilkesini yakalıyor ama cevap kesilmiş, aykırılığı net söylemiyor.
    0.5,  # 2 - Yaşama hakkı bağlamı var ama "Hayır" diyerek sonucu ters veriyor.
    0.0,  # 3 - Geçici madde 20 tarihi yanlış; corpus coverage problemi.
    0.5,  # 4 - TCK source'a gitmiş ve "sabit değil" fikri var ama bilirkişi/dosya incelemesi yok.
    0.0,  # 5 - Cumhurbaşkanı yemini cevabı hâlâ alakasız/eksik.
    0.0,  # 6 - Anayasa 122 yoktur diyor, yanlış.
    1.0,  # 7 - Yedi gün doğru.
    0.5,  # 8 - "Evet" doğru yön ama açıklama karışık/eksik.
    0.0,  # 9 - DDK ve Silahlı Kuvvetler istisnasını doğru vermiyor.
    1.0,  # 10 - 13. madde için aykırıdır ve kanunla sınırlama gerekçesi doğru.
    0.0,  # 11 - KVKK'daki ilgili kişi tanımı corpus'ta yok; cevap yanlış.
    0.5,  # 12 - AYM kararlarının kesin uygulanması fikri var ama kesinlik anlamı eksik.
    1.0,  # 13 - Madde 63 doğru.
    0.5,  # 14 - Madde 47 kısmen doğru ama eksik/karışık.
    0.5,  # 15 - "Kanunla düzenlenir" kısmen doğru ama KVKK ve diğer kanunlar net değil.
    0.5,  # 16 - Kanunla düzenlenir ana fikri doğru ama detaylar eksik.
    0.0,  # 17 - 115 yerine 109 demiş, yanlış.
    1.0,  # 18 - Bilgi Edinme Değerlendirme Kurulu'na başvuru fikri doğru yakalanmış.
    0.0   # 19 - Invalid sample, skora katılmayacak.
]

source_aware_test_results_df["manual_score"] = manual_scores_source_aware

valid_source_aware_df = source_aware_test_results_df[
    source_aware_test_results_df["is_valid_sample"] == True
]

source_aware_test_score = valid_source_aware_df["manual_score"].mean()

print("Source-Aware + Article-Aware + Turkish BGE RAG Test Score:", source_aware_test_score)
print("Valid sample count:", len(valid_source_aware_df))
print("Total sample count:", len(source_aware_test_results_df))

Source-Aware + Article-Aware + Turkish BGE RAG Test Score: 0.42105263157894735
Valid sample count: 19
Total sample count: 20


In [75]:
source_aware_test_results_df.to_csv(
    f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Source-Aware + Article-Aware Retrieval + Turkish BGE Reranker Fusion + Improved Prompt",
    "manual_accuracy": source_aware_test_score,
    "valid_sample_count": len(valid_source_aware_df),
    "total_sample_count": len(source_aware_test_results_df),
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT,
    "article_bonus_weight": ARTICLE_BONUS_WEIGHT
}]).to_csv(
    f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Source-aware scored results saved.")

Source-aware scored results saved.


In [76]:
test_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Base prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "Strict Prompt RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "FlashRank Fusion Reranker RAG",
        "retrieval_setup": "Hybrid top-10 + FlashRank reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.184211
    },
    {
        "method": "Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.315789
    },
    {
        "method": "Improved Prompt + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": 0.394737
    },
    {
        "method": "Article-Aware Retrieval + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + article bonus + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": 0.394737
    },
    {
        "method": "Source-Aware + Article-Aware Retrieval + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + source routing + article bonus + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": source_aware_test_score
    }
])

test_comparison_df

,method,retrieval_setup,prompt,manual_accuracy
0,Base RAG,"Hybrid retrieval top-5, context top-3",Base prompt,0.263158
1,Strict Prompt RAG,"Hybrid retrieval top-5, context top-3",Strict prompt,0.263158
2,FlashRank Fusion Reranker RAG,Hybrid top-10 + FlashRank reranker + rank fusi...,Strict prompt,0.184211
3,Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Strict prompt,0.315789
4,Improved Prompt + Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Improved legal prompt,0.394737
5,Article-Aware Retrieval + Turkish BGE Reranker...,Hybrid top-10 + article bonus + Turkish BGE re...,Improved legal prompt,0.394737
6,Source-Aware + Article-Aware Retrieval + Turki...,Hybrid top-10 + source routing + article bonus...,Improved legal prompt,0.421053


In [77]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_source_aware_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Source-aware comparison saved.")

Source-aware comparison saved.
